In [1]:
import pandas as pd
from pathlib import Path

In [2]:
ventas = pd.read_csv(Path(r"D:\ProyectoAnalisisElectrico\TransferenciasEconomicas\Ventas_SEN_2505_2604.csv"), sep=",")
calor = pd.read_csv(Path(r"D:\ProyectoAnalisisElectrico\CupraThermV2\Resumen_Establecimientos.csv"), sep=",")

In [3]:
calor.columns

Index(['CODIGO_VU', 'RAZON_SOCIAL', 'RUT_RAZON_SOCIAL',
       'NOMBRE_ESTABLECIMIENTO', 'COMBUSTIBLE_PRIMARIO', 'LATITUD', 'LONGITUD',
       'REGION', 'RUBRO', 'CIIU6', 'DEMANDA_CALOR_MWH', 'CALNUM', 'Acrolein',
       'Arsenic', 'Benzene', 'Cadmium', 'Carbon dioxide', 'Carbon monoxide',
       'Formaldehyde', 'Lead', 'Mercury', 'Methane', 'Nitrogen oxides (NOx)',
       'Nitrous oxide', 'Pentachlorophenol (PCP)', 'Sulfur dioxide', 'Toluene',
       'Volatile organic compounds (VOC)', 'Black Carbon', 'PCDD-F',
       'Sulfur oxides (SOx)', 'Ammonia', 'PM, primary', 'PM10, primary',
       'PM2.5, primary'],
      dtype='str')

In [14]:
calor["REGION"].unique()

<ArrowStringArray>
[              'Metropolitana de Santiago',
                               'Los Lagos',
                             'Antofagasta',
                                'Tarapacá',
     'Libertador Gral. Bernardo O'Higgins',
                              'Valparaíso',
                                   'Maule',
                                   'Ñuble',
                                  'Biobío',
                               'Araucanía',
                      'Arica y Parinacota',
    'Magallanes y de la Antártica Chilena',
                                'Coquimbo',
                                'Los Ríos',
                                 'Atacama',
 'Aysén del Gral. Carlos Ibañez del Campo']
Length: 16, dtype: str

In [4]:
calor["COMBUSTIBLE_PRIMARIO"].unique()

<ArrowStringArray>
[               'Gas Natural',    'Gas Licuado de Petróleo',
                     'Varios',      'Petróleo N 2 (Diesel)',
               'Petróleo N 6',          'Carbón Bituminoso',
    'MDO (Marine Diesel Oil)',               'Petróleo N 5',
 'Coke de Petróleo (Petcoke)',             'Gas de Cañería',
                    'Propano']
Length: 11, dtype: str

In [5]:
# Diccionario para estandarizar nombres de regiones (corrige tildes, mayúsculas y variaciones tipográficas)
mapeo_regiones = {
    "Metropolitana De Santiago": "Metropolitana de Santiago",
    "Libertad Bernardo O'Higgins": "Libertad Gral. Bernardo O'Higgins",
    "La Araucania": "Araucanía",
    "La Araucanía": "Araucanía",
    "Bío Bío": "Biobío",
    "Biobío": "Biobío" 
}

# Aplica el mapeo para crear una columna geográficamente limpia, asegurando que las futuras agrupaciones por región sean correctas
ventas["REGION_HOMOLOGADA"] = ventas["REGION"].replace(mapeo_regiones)

In [ ]:
# Define las reglas estadísticas para la demanda de calor y extrae valores únicos para categorizar
agregaciones_calor = {
    'DEMANDA_CALOR_MWH': ['sum', 'mean', 'std', 'max', 'min'],
    'NOMBRE_ESTABLECIMIENTO': lambda x: list(set(x.dropna())),
    'COMBUSTIBLE_PRIMARIO': lambda x: list(set(x.dropna())),
    'RUBRO': lambda x: list(set(x.dropna())),
    'DEMANDA_CALOR_MWH': lambda x: list(set(x.dropna()))
}

# Consolida la información a nivel de empresa (RUT_RAZON_SOCIAL) por región (hub industrial)
df_calor_hubs = calor.groupby(
    ['RUT_RAZON_SOCIAL', 'REGION']
).agg(agregaciones_calor).reset_index()

# Aplana el MultiIndex generado por agg() (ej. de ('DEMANDA_CALOR_MWH', 'sum') a 'DEMANDA_CALOR_MWH_sum')
df_calor_hubs.columns = [
    '_'.join(col).strip('_') if type(col) is tuple else col 
    for col in df_calor_hubs.columns.values
]

# Limpia los sufijos '<lambda>' que pandas asigna automáticamente a las funciones personalizadas
df_calor_hubs = df_calor_hubs.rename(columns={
    'NOMBRE_ESTABLECIMIENTO_<lambda>': 'NOMBRE_ESTABLECIMIENTO',
    'COMBUSTIBLE_PRIMARIO_<lambda>': 'COMBUSTIBLE_PRIMARIO',
    'RUBRO_<lambda>': 'RUBRO'
})

columnas_texto = ['NOMBRE_ESTABLECIMIENTO', 'COMBUSTIBLE_PRIMARIO', 'RUBRO']

# Transforma las listas de valores únicos en un solo string separado por '::' para facilitar su exportación
for col in columnas_texto:
    df_calor_hubs[col] = df_calor_hubs[col].apply(lambda x: '::'.join(map(str, x)))

# Reemplaza los nulos por 0 en la desviación estándar (situación que ocurre cuando un hub tiene un solo registro)
df_calor_hubs['DEMANDA_CALOR_MWH_std'] = df_calor_hubs['DEMANDA_CALOR_MWH_std'].fillna(0)

In [7]:
cruce_calor_ventas = pd.merge(df_calor_hubs, ventas, how="inner", left_on=["RUT_RAZON_SOCIAL", "REGION"], right_on=["RUTCLIENTE", "REGION_HOMOLOGADA"])

In [8]:
# Define un orden lógico y estructurado para presentar el resultado del cruce entre las bases de calor y ventas
nuevo_orden = [
    # 1. Identificadores principales, llaves de cruce y variables temporales
    'CLIENTE2', 
    'RUTCLIENTE', 
    'TIPO', 
    'clave', 
    'NOMBRE_ESTABLECIMIENTO', 
    'REGION_HOMOLOGADA', 
    'COMBUSTIBLE_PRIMARIO', 
    'AÑO', 
    'MES',
    
    # 2. Atributos corporativos e industriales
    'RUT_RAZON_SOCIAL', 
    'RUBRO', 
    
    # 3. Trazabilidad geográfica (mantiene el registro de ambas fuentes originales para auditoría)
    'REGION_x', # Región proveniente de la base de calor
    'REGION_y', # Región proveniente de la base de ventas
    
    # 4. Detalles topológicos de conexión eléctrica, sectores y responsabilidades en balances de transferencias
    'CLIENTE', 
    'SECTOR', 
    'SUBSECTOR', 
    'PUNTO DE CONEXIÓN', 
    'ID_PC',
    'RUTCLIENTEPUNTO DE CONEXIÓN', 
    'CLIENTE SUMINISTRADO POR_O', 
    'CLIENTE SUMINISTRADO POR', 
    'RECONOCE EL RETIRO EN BALANCES DE TRANSFERENCIAS_O', 
    'RECONOCE EL RETIRO EN BALANCES DE TRANSFERENCIAS', 
    'RETIRO', 
    
    # 5. Métricas operativas consolidadas (consumo eléctrico y estadísticas de demanda térmica)
    'ENERGIA2',
    'DEMANDA_CALOR_MWH_sum', 
    'DEMANDA_CALOR_MWH_mean', 
    'DEMANDA_CALOR_MWH_std', 
    'DEMANDA_CALOR_MWH_max', 
    'DEMANDA_CALOR_MWH_min',
    
    # 6. Variables auxiliares o de control interno
    'col1', 
    'col2'
]

# Reorganiza el DataFrame aplicando el orden definido para facilitar su lectura y exportación final
cruce_calor_ventas = cruce_calor_ventas[nuevo_orden]

In [9]:
# Construye un identificador temporal unificado en formato 'YYMM' para facilitar cruces temporales.
# Se fuerza el tipo entero previo a la conversión a texto para evitar arrastrar decimales (.0)
año_str = cruce_calor_ventas['AÑO'].astype(int).astype(str).str[-2:]
mes_str = cruce_calor_ventas['MES'].astype(int).astype(str).str.zfill(2)

cruce_calor_ventas['PERIODO'] = año_str + mes_str

# Define el subconjunto definitivo de variables relevantes para el análisis y reportabilidad
columnas_finales = [
    'clave', 
    'CLIENTE', 
    'TIPO', 
    'NOMBRE_ESTABLECIMIENTO', 
    'DEMANDA_CALOR_MWH_sum', 
    'DEMANDA_CALOR_MWH_mean', 
    'DEMANDA_CALOR_MWH_std', 
    'DEMANDA_CALOR_MWH_max', 
    'DEMANDA_CALOR_MWH_min', 
    'RUT_RAZON_SOCIAL', 
    'REGION_HOMOLOGADA', 
    'COMBUSTIBLE_PRIMARIO',
    'PERIODO', # Reemplaza a las columnas individuales de AÑO y MES
    'RUBRO',
    'SECTOR', 
    'SUBSECTOR'
]

# Aísla la tabla final en un nuevo DataFrame (copy) para proteger los datos originales 
# y evitar advertencias de SettingWithCopy de pandas al modificar el set limpio
cruce_calor_ventas_limpio = cruce_calor_ventas[columnas_finales].copy()

In [10]:
cruce_calor_ventas_limpio.rename(columns={"REGION_HOMOLOGADA": "REGION"}, inplace=True)

In [11]:
# Mapeo oficial para agrupar las regiones de Chile en macrozonas geográficas.
# Esta agregación es útil para análisis espaciales a gran escala, como el estudio 
# de nudos de consumo o el comportamiento regional de la red eléctrica.
macrozonas = {
    # 1.- Macrozona Norte Grande
    'Arica y Parinacota': 'Norte Grande',
    'Tarapacá': 'Norte Grande',
    'Antofagasta': 'Norte Grande',

    # 2.- Macrozona Norte Chico
    'Atacama': 'Norte Chico',
    'Coquimbo': 'Norte Chico',

    # 3.- Macrozona Centro
    'Valparaíso': 'Centro',
    'Metropolitana de Santiago': 'Centro', 

    # 4.- Macrozona Centro Sur
    'Libertador Gral. Bernardo O\'Higgins': 'Centro Sur',
    'Maule': 'Centro Sur',
    'Ñuble': 'Centro Sur',
    'Biobío': 'Centro Sur',

    # 5.- Macrozona Sur
    # El string 'Araucanía' coincide exactamente con el resultado de la limpieza de datos anterior
    'Araucanía': 'Sur',  
    'Los Ríos': 'Sur',
    'Los Lagos': 'Sur',

    # 6.- Macrozona Austral
    'Aysén del General Carlos Ibáñez del Campo': 'Austral',
    'Magallanes y de la Antártica Chilena': 'Austral'
}

# Crea una nueva dimensión analítica mapeando la región de cada instalación a su respectiva macrozona
cruce_calor_ventas_limpio['macrozona'] = cruce_calor_ventas_limpio['REGION'].map(macrozonas)

In [12]:
cruce_calor_ventas_limpio.to_csv(Path(r"D:\ProyectoAnalisisElectrico\PotencialesClientes\CalorVentasRegionales.csv"), index=False)